# Template code for running MAST to compare clones
Requires conda environment with rpy2; outputs are provided in data directory

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np 
import pandas as pd

import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
large_data_dir = gf_utils.large_data_dir


In [ ]:
import rpy2
import anndata2ri
import os
import scanpy as sc
import pandas as pd
import numpy as np

# anndata2ri interconverts AnnData and Single Cell Experiment objects
anndata2ri.activate()
%load_ext rpy2.ipython
#%reload_ext rpy2.ipython

import matplotlib.pyplot as plt
import scipy.stats as st

import seaborn as sns
import glob
from functools import reduce


In [ ]:
adata_dir = large_data_dir + 'MPN_WTA/MPN_1_BC007_genotyped.h5ad'
adata = sc.read_h5ad(adata_dir)

adata.obs['cell_type'] = adata.obs['cell_type'].cat.add_categories(['leukemic blast'])
adata.uns['cell_type_colors'] = np.append(adata.uns['cell_type_colors'], ['#000000'])
adata.obs.loc[adata.obs['pheno_leiden'].isin([4,9,16,13,15,17]) & ~(adata.obs['cell_type'].isin(['B cell (non-HSPC)', 'T cell (non-HSPC)'])), 'cell_type'] = 'leukemic blast'

adata.obs['clone'] = pd.read_csv('../output/clone_assignments.csv', index_col=0)['clone']

In [ ]:
sc.pl.umap(adata[adata.obs['pheno_leiden'].isin([12,1,8])], color = 'clone')
sc.pl.umap(adata[adata.obs['pheno_leiden'].isin([12,1,8])], color = 'pheno_leiden')

In [ ]:
%%capture

adata = adata[adata.obs['pheno_leiden'].isin([12,1,8])].copy()

for cluster in adata.obs['pheno_leiden'].dropna().unique():
            
    output_filename = f"pheno_leiden_{cluster}_vs_all_mast_de"

    print(f"Processing cluster: {cluster}")

    adata.obs['in_type'] = (adata.obs['pheno_leiden'] == cluster).map({True: 'in', False: 'out'})

    sce_v2 = sc.AnnData(X = adata.X, 
                    obs = pd.DataFrame({'in_type': adata.obs['in_type']}, 
                                        index = adata.obs.index),
                    var = pd.DataFrame(index = adata.var.index))
    
    %R library(scuttle)
    %R library(MAST)
    %R library(data.table)

    # -i implies we are supplying sce_v2 as an input to R
    %R -i sce_v2 -i output_filename

    # set up a single cell experiment object in R, using the data stored in 'X' in sce anndata object
    %R counts(sce_v2) <- assay(sce_v2, "X"); 
    print("Finished Setup")

    ### run MAST
    # identify in cells:
    %R id_cells_in <- which(colData(sce_v2)$in_type == 'in')

    # identify out cells:
    %R id_cells_out <- which(colData(sce_v2)$in_type == 'out')

    # Check the length of the two sets
    %R print(length(id_cells_in))
    %R print(length(id_cells_out))

    # Create two dataframes: one with in cells and one with out cells
    %R df1 <- t(data.frame(counts(sce_v2)[, id_cells_out])) # transpose because in sce genes are rows
    %R df2 <- t(data.frame(counts(sce_v2)[, id_cells_in])) # transpose because in sce genes are rows

    # We will use the function provided:
    %R source("run_MAST.r")

    # Syntax: %R pairwise_de(dataframe1, dataframe2, 'output_filename', 'output_folder')
    # The output will be automatically saved as a csv file in the output_folder with output_filename
    %R pairwise_de(df2, df1, output_filename, '../data/')